# End-to-End Plan for Fine-Tuning BERT for Record Linkage

## Step 1: Data Preparation & Splitting

**Goal:** Create clean, correctly formatted, and separated datasets for training, validation, and testing, while preventing data leakage.

1.  **Group Entities**:
    * Load your list of labeled matching pairs.
    * Construct a graph where records are nodes and matches are edges.
    * Find the **connected components** of this graph. Each component represents a single real-world entity (e.g., `{record_A1, record_B1, record_B2}`).

2.  **Perform Entity-Based Split**:
    * Create a list of all unique entity clusters.
    * Randomly shuffle and split this list of *entities* into three sets: 80% for training, 10% for validation, and 10% for testing.

3.  **Construct Datasets**:
    * Create the final train, validation, and test record sets based on the entity split. Ensure no entity appears in more than one set.

4.  **Augment Training Pairs**:
    * For the **training set only**, iterate through each entity cluster.
    * If an entity has more than two records (e.g., `{a, b1, b2}`), generate all possible unique positive pairs from within that cluster (e.g., `(a, b1)`, `(a, b2)`, and `(b1, b2)`). This provides a richer training signal.

5.  **Serialize All Records**:
    * Write a function to convert each structured record (row) into a single string using the `[COL] attribute [VAL] value` format.
    * Apply this serialization to every record in the train, validation, and test sets.

## Step 2: Establish Baseline (Zero-Shot Evaluation)

**Goal:** Measure the performance of the pre-trained model *before* any fine-tuning to have a benchmark for comparison.

1.  **Select Model**: Choose a pre-trained sentence-transformer model suitable for structured data (e.g., `sentence-transformers/gtr-t5-base`).
2.  **Generate Embeddings**: Load the serialized records from your **test set** and pass them through the base model to get a vector embedding for each.
3.  **Perform Similarity Search**:
    * Build a FAISS index on the embeddings from one table in the test set.
    * Query the index with the embeddings from the other table to find the top-1 nearest neighbor for each record.
4.  **Calculate Baseline Metrics**: Compare the predicted pairs from FAISS against the ground truth labels for the test set. Calculate **Precision, Recall, and F1-Score**. This is your zero-shot baseline.

## Step 3: Fine-Tune the Model

1.  **Prepare DataLoader**: Load the augmented and serialized training pairs into a PyTorch DataLoader, which will feed batches to the model.
2.  **Define Loss Function**: Instantiate the `MultipleNegativesRankingLoss`.
3.  **Set Up Evaluator**: Configure an evaluator that runs periodically during training. It should use the **validation set** to compute a relevant metric (e.g., cosine similarity F1-score) to track performance on unseen data.
4.  **Train**: Run the model's `fit()` method, pointing it to your training data, loss function, and evaluator. The training process should save the model checkpoint that performs best on the validation set.

## Step 4: Find the Optimal Classification Threshold

**Goal:** Determine the best similarity score cutoff to classify a pair as a "match" or "non-match".

1.  **Load Best Model**: Load the fine-tuned model checkpoint that achieved the best performance on the validation set during training.
2.  **Generate Validation Embeddings**: Use this model to create embeddings for all records in your **validation set**.
3.  **Calculate Scores**: Compute the cosine similarity scores for all known positive and known negative pairs within the validation set.
4.  **Iterate and Evaluate**: Loop through a range of possible thresholds (e.g., from 0.50 to 1.00 in steps of 0.01). For each threshold, classify the pairs and calculate the F1-score.
5.  **Select Threshold**: Identify the threshold that resulted in the highest F1-score. This is your **optimal threshold**.

## Step 5: Final Evaluation

In [ ]:
import pandas as pd
import numpy as np
import networkx as nx
from collections import defaultdict
import pickle
import os
from sklearn.model_selection import train_test_split
from itertools import combinations
import random

# Set random seed for reproducibility
random.seed(42)
np.random.seed(42)

In [ ]:
# Load the datasets
matches_df = pd.read_csv('../../data/dblp-scholar/matches.csv')
dblp_df = pd.read_csv('../../data/dblp-scholar/dblp.csv')
scholar_df = pd.read_csv('../../data/dblp-scholar/scholar.csv')

print(f"Loaded {len(matches_df)} matching pairs")
print(f"Loaded {len(dblp_df)} DBLP records")
print(f"Loaded {len(scholar_df)} Scholar records")
print(f"\nMatches sample:")
print(matches_df.head())
print(f"\nDBLP sample:")
print(dblp_df.head())
print(f"\nScholar sample:")
print(scholar_df.head())


In [ ]:
def build_entity_graph(matches_df):
    """
    Build a graph from matching pairs and find connected components (entities).
    Each connected component represents a single real-world entity.
    """
    # Create a graph
    G = nx.Graph()
    
    # Add edges for each matching pair
    for _, row in matches_df.iterrows():
        dblp_id = row['idDBLP']
        scholar_id = row['idScholar']
        G.add_edge(dblp_id, scholar_id)
    
    # Find connected components (entity clusters)
    entity_clusters = list(nx.connected_components(G))
    
    print(f"Found {len(entity_clusters)} entity clusters")
    print(f"Cluster size distribution:")
    
    cluster_sizes = [len(cluster) for cluster in entity_clusters]
    size_counts = defaultdict(int)
    for size in cluster_sizes:
        size_counts[size] += 1
    
    for size in sorted(size_counts.keys()):
        print(f"  Size {size}: {size_counts[size]} clusters")
    
    return entity_clusters

# Build the entity graph
entity_clusters = build_entity_graph(matches_df)


In [ ]:
def perform_entity_based_split(entity_clusters, train_ratio=0.8, val_ratio=0.1, test_ratio=0.1):
    """
    Split entity clusters into train/val/test sets ensuring no entity appears in multiple sets.
    """
    assert abs(train_ratio + val_ratio + test_ratio - 1.0) < 1e-6, "Ratios must sum to 1.0"
    
    # Convert to list and shuffle
    clusters_list = list(entity_clusters)
    random.shuffle(clusters_list)
    
    n_clusters = len(clusters_list)
    n_train = int(n_clusters * train_ratio)
    n_val = int(n_clusters * val_ratio)
    
    # Split clusters
    train_clusters = clusters_list[:n_train]
    val_clusters = clusters_list[n_train:n_train + n_val]
    test_clusters = clusters_list[n_train + n_val:]
    
    print(f"Entity split:")
    print(f"  Train: {len(train_clusters)} clusters")
    print(f"  Validation: {len(val_clusters)} clusters")
    print(f"  Test: {len(test_clusters)} clusters")
    
    return train_clusters, val_clusters, test_clusters

# Perform entity-based split
train_clusters, val_clusters, test_clusters = perform_entity_based_split(entity_clusters)


In [ ]:
def construct_datasets(train_clusters, val_clusters, test_clusters, dblp_df, scholar_df):
    """
    Construct final train/val/test datasets based on entity clusters.
    Returns records from both datasets that belong to each split.
    """
    # Create mappings for quick lookup
    dblp_records = dblp_df.set_index('id').to_dict('index')
    scholar_records = scholar_df.set_index('id').to_dict('index')
    
    def extract_records_from_clusters(clusters):
        """Extract all records (DBLP and Scholar) from given clusters"""
        records = []
        for cluster in clusters:
            for record_id in cluster:
                # Check if it's a DBLP record
                if record_id in dblp_records:
                    record = dblp_records[record_id].copy()
                    record['id'] = record_id
                    record['source'] = 'dblp'
                    records.append(record)
                # Check if it's a Scholar record
                elif record_id in scholar_records:
                    record = scholar_records[record_id].copy()
                    record['id'] = record_id
                    record['source'] = 'scholar'
                    records.append(record)
        return pd.DataFrame(records)
    
    # Extract records for each split
    train_records = extract_records_from_clusters(train_clusters)
    val_records = extract_records_from_clusters(val_clusters)
    test_records = extract_records_from_clusters(test_clusters)
    
    print(f"Dataset construction:")
    print(f"  Train records: {len(train_records)} ({len([r for r in train_records['source'] if r == 'dblp'])} DBLP, {len([r for r in train_records['source'] if r == 'scholar'])} Scholar)")
    print(f"  Val records: {len(val_records)} ({len([r for r in val_records['source'] if r == 'dblp'])} DBLP, {len([r for r in val_records['source'] if r == 'scholar'])} Scholar)")
    print(f"  Test records: {len(test_records)} ({len([r for r in test_records['source'] if r == 'dblp'])} DBLP, {len([r for r in test_records['source'] if r == 'scholar'])} Scholar)")
    
    return train_records, val_records, test_records, train_clusters, val_clusters, test_clusters

# Construct the datasets
train_records, val_records, test_records, train_clusters, val_clusters, test_clusters = construct_datasets(
    train_clusters, val_clusters, test_clusters, dblp_df, scholar_df
)


In [ ]:
def generate_training_pairs(train_clusters):
    """
    Generate all possible positive pairs within each training cluster.
    For clusters with more than 2 records, this creates additional training examples.
    """
    training_pairs = []
    
    for cluster in train_clusters:
        cluster_list = list(cluster)
        # Generate all possible pairs within the cluster
        if len(cluster_list) >= 2:
            for pair in combinations(cluster_list, 2):
                training_pairs.append({
                    'record_1': pair[0],
                    'record_2': pair[1],
                    'label': 1  # Positive pair
                })
    
    print(f"Generated {len(training_pairs)} positive training pairs")
    
    # Show distribution of cluster sizes that contributed to pairs
    cluster_sizes = [len(cluster) for cluster in train_clusters if len(cluster) >= 2]
    size_counts = defaultdict(int)
    for size in cluster_sizes:
        size_counts[size] += 1
    
    print("Cluster size distribution in training:")
    for size in sorted(size_counts.keys()):
        pairs_from_size = size * (size - 1) // 2  # Combinations formula
        total_pairs = size_counts[size] * pairs_from_size
        print(f"  Size {size}: {size_counts[size]} clusters -> {total_pairs} pairs")
    
    return training_pairs

# Generate training pairs
training_pairs = generate_training_pairs(train_clusters)


In [ ]:
def serialize_record(record):
    """
    Convert a structured record into a serialized string using [COL] attribute [VAL] value format.
    """
    serialized_parts = []
    
    # Define the order of columns to serialize
    columns = ['title', 'authors', 'venue', 'year']
    
    for col in columns:
        if col in record and pd.notna(record[col]) and str(record[col]).strip():
            value = str(record[col]).strip()
            serialized_parts.append(f"[COL] {col} [VAL] {value}")
    
    return " ".join(serialized_parts)

def serialize_dataset(records_df):
    """
    Apply serialization to all records in a dataset.
    """
    serialized_records = {}
    
    for _, record in records_df.iterrows():
        record_id = record['id']
        serialized_text = serialize_record(record)
        serialized_records[record_id] = {
            'id': record_id,
            'source': record['source'],
            'serialized_text': serialized_text,
            'original_record': record.to_dict()
        }
    
    return serialized_records

# Serialize all datasets
print("Serializing datasets...")
train_serialized = serialize_dataset(train_records)
val_serialized = serialize_dataset(val_records)
test_serialized = serialize_dataset(test_records)

print(f"Serialized {len(train_serialized)} training records")
print(f"Serialized {len(val_serialized)} validation records")
print(f"Serialized {len(test_serialized)} test records")

# Show some examples
print("\nSerialization examples:")
sample_ids = list(train_serialized.keys())[:3]
for i, record_id in enumerate(sample_ids):
    record = train_serialized[record_id]
    print(f"\nExample {i+1} ({record['source']}):")
    print(f"ID: {record_id}")
    print(f"Serialized: {record['serialized_text']}")


In [ ]:
# Create final training pairs with serialized texts
def create_final_training_pairs(training_pairs, train_serialized):
    """
    Create final training pairs with serialized texts for BERT training.
    """
    final_pairs = []
    
    for pair in training_pairs:
        record_1_id = pair['record_1']
        record_2_id = pair['record_2']
        
        if record_1_id in train_serialized and record_2_id in train_serialized:
            final_pairs.append({
                'text_1': train_serialized[record_1_id]['serialized_text'],
                'text_2': train_serialized[record_2_id]['serialized_text'],
                'record_1_id': record_1_id,
                'record_2_id': record_2_id,
                'label': pair['label']
            })
    
    return final_pairs

final_training_pairs = create_final_training_pairs(training_pairs, train_serialized)
print(f"Created {len(final_training_pairs)} final training pairs with serialized texts")

# Show a training pair example
if final_training_pairs:
    example = final_training_pairs[0]
    print(f"\nTraining pair example:")
    print(f"Record 1 ID: {example['record_1_id']}")
    print(f"Text 1: {example['text_1']}")
    print(f"Record 2 ID: {example['record_2_id']}")
    print(f"Text 2: {example['text_2']}")
    print(f"Label: {example['label']}")


In [ ]:
# Save all processed data for future use
print("Saving processed datasets...")

# Create directory for processed data
os.makedirs('processed_data', exist_ok=True)

# Save serialized records
with open('processed_data/train_serialized.pkl', 'wb') as f:
    pickle.dump(train_serialized, f)

with open('processed_data/val_serialized.pkl', 'wb') as f:
    pickle.dump(val_serialized, f)

with open('processed_data/test_serialized.pkl', 'wb') as f:
    pickle.dump(test_serialized, f)

# Save training pairs
with open('processed_data/training_pairs.pkl', 'wb') as f:
    pickle.dump(final_training_pairs, f)

# Save cluster information
with open('processed_data/clusters.pkl', 'wb') as f:
    pickle.dump({
        'train_clusters': train_clusters,
        'val_clusters': val_clusters,
        'test_clusters': test_clusters
    }, f)

print("Saved processed data to processed_data/ directory:")
print(f"  - train_serialized.pkl: {len(train_serialized)} records")
print(f"  - val_serialized.pkl: {len(val_serialized)} records")
print(f"  - test_serialized.pkl: {len(test_serialized)} records")
print(f"  - training_pairs.pkl: {len(final_training_pairs)} pairs")
print(f"  - clusters.pkl: cluster information")


In [ ]:
# Data quality and leakage verification
def verify_no_data_leakage(train_clusters, val_clusters, test_clusters):
    """
    Verify that no entity appears in multiple datasets.
    """
    train_records = set()
    val_records = set()
    test_records = set()
    
    for cluster in train_clusters:
        train_records.update(cluster)
    
    for cluster in val_clusters:
        val_records.update(cluster)
    
    for cluster in test_clusters:
        test_records.update(cluster)
    
    # Check for overlaps
    train_val_overlap = train_records.intersection(val_records)
    train_test_overlap = train_records.intersection(test_records)
    val_test_overlap = val_records.intersection(test_records)
    
    print("Data leakage verification:")
    print(f"  Train-Val overlap: {len(train_val_overlap)} records")
    print(f"  Train-Test overlap: {len(train_test_overlap)} records")
    print(f"  Val-Test overlap: {len(val_test_overlap)} records")
    
    if len(train_val_overlap) == 0 and len(train_test_overlap) == 0 and len(val_test_overlap) == 0:
        print("  ✅ No data leakage detected!")
    else:
        print("  ❌ Data leakage detected!")
        
    return len(train_val_overlap) == 0 and len(train_test_overlap) == 0 and len(val_test_overlap) == 0

# Verify no data leakage
leakage_check = verify_no_data_leakage(train_clusters, val_clusters, test_clusters)

# Summary statistics
print(f"\n📊 Final Dataset Summary:")
print(f"  Total entities: {len(train_clusters) + len(val_clusters) + len(test_clusters)}")
print(f"  Training entities: {len(train_clusters)} ({len(train_clusters)/(len(train_clusters) + len(val_clusters) + len(test_clusters))*100:.1f}%)")
print(f"  Validation entities: {len(val_clusters)} ({len(val_clusters)/(len(train_clusters) + len(val_clusters) + len(test_clusters))*100:.1f}%)")
print(f"  Test entities: {len(test_clusters)} ({len(test_clusters)/(len(train_clusters) + len(val_clusters) + len(test_clusters))*100:.1f}%)")
print(f"  Training records: {len(train_serialized)}")
print(f"  Validation records: {len(val_serialized)}")
print(f"  Test records: {len(test_serialized)}")
print(f"  Training pairs: {len(final_training_pairs)}")
print(f"  Data leakage: {'None' if leakage_check else 'Detected'}")
print(f"\n✅ Data preparation completed successfully!")


**Goal:** Measure the final performance of your fine-tuned model on completely unseen data.

1.  **Generate Test Embeddings**: Use your best fine-tuned model to generate embeddings for all records in the **test set**.
2.  **Classify Pairs**: Find the nearest neighbor for each record (using FAISS or brute-force) and calculate the cosine similarity score. Apply your **optimal threshold** to classify each pair as a match or non-match.
3.  **Calculate Final Metrics**: Compare your model's final predictions against the ground truth labels for the test set. Calculate the final **Precision, Recall, and F1-Score**.
4.  **Compare**: Compare these final scores to the baseline from Step 2 to quantify the improvement from fine-tuning.